In [1]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn matplotlib catboost optuna plotly nbformat

In [ ]:
import sys
from pathlib import Path

# adiciona a pasta src ao sys.path
SRC_DIR = Path().resolve().parent  # .../src
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

In [2]:
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline

from catboost import CatBoostClassifier

import optuna

from utils.constants import *

In [3]:
df = pd.read_csv("../data/3_gold/dataset-processed-gb.csv")

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [12]:
FIXED_PARAMS = {
    'iterations': 2000,
    'loss_function': 'MultiClass',
    'eval_metric': 'AUC',         
    'auto_class_weights': 'Balanced',
    'early_stopping_rounds': 5,
    'random_state': RANDOM_STATE,
    'cat_features': categorical_features,
    'allow_writing_files': False
}

In [13]:
# def get_pipeline(params, class_counts=None):
#     steps = []

#     if class_counts:
#         n_low, n_alarm, n_severe = class_counts[0], class_counts[1], class_counts[2]
#         steps.append(('under', RandomUnderSampler(sampling_strategy={0: max(n_alarm, n_low // 2)}, random_state=RANDOM_STATE)))
#         steps.append(('over', SMOTENC(categorical_features=categorical_features, sampling_strategy={1: max(n_severe, n_alarm // 2)}, random_state=RANDOM_STATE)))
    
#     steps.append(('classifier', CatBoostClassifier(**params)))
#     return Pipeline(steps=steps)


def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        # clf = get_pipeline(params)

        clf = CatBoostClassifier(**params)

        try:
            clf.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)
            preds = clf.predict(X_valid)
            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0
        
    return np.mean(scores), np.std(scores)


def objective(trial: optuna.trial.Trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
        'depth': trial.suggest_int('depth', 5, 10, step=1),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.1, 1.0, step=0.1),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        **FIXED_PARAMS
    }

    if params['bootstrap_type'] == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
    elif params['bootstrap_type'] == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.1, 1.0, log=True)

    avg, _ = train_on_folds(params)
    return avg

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

# From run on Colab:
# Best F1: 0.5093228162486619
# Best Params: {'learning_rate': 0.03158826697434671, 'depth': 8, 'colsample_bylevel': 0.6, 'min_data_in_leaf': 19, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.1985479799912282}

[I 2026-02-06 11:37:44,054] A new study created in memory with name: no-name-391f3610-b3da-44ca-a085-1eff8e6b8563


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import pickle
from pathlib import Path

output_dir = Path('../results/optimization/catboost')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [ ]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [ ]:
avg_f1_final, std_f1_final = train_on_folds(best_trial.params)